In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

#     "axes.labelsize": 12,
#     "legend.fontsize": 10,

OPT_COLORS = {
    'NSGA2': '#d62728',  # strong red (best)
    'BO': '#1f77b4',     # blue
    'LHS': '#2ca02c',    # green
    'Grid': '#7f7f7f'    # gray (baseline → visually de-emphasized)      'Grid': '#9e9e9e'     # muted baseline (IMPORTANT change)
}

OPT_COLORS1 = {
    'NSGA2': '#DC4748',
    'BO': '#84B4D5',
    'LHS': '#B5DDB5',
    'Grid': '#D2D2D2'
}

# c_csh = '#1b4f72'      # deep blue
# c_t14 = '#a93226'      # muted red
# c_t11m = '#1e8449'     # muted green
# c_t11h = '#6c3483'     # muted purple

In [ ]:
# Load data
df = pd.read_csv("../../../data/reference_results/error_data_pcff_all_wi_SE.csv")

# Remove unwanted columns
df = df.loc[:, ~df.columns.str.contains('Unnamed')]

# Add iteration index
df['iteration'] = df.groupby('optimizers', sort=False).cumcount() + 1

In [ ]:
# Total error (all 9 objectives)
df['total_error'] = df[[col for col in df.columns if col.startswith('error_')]].sum(axis=1)

# Grouped errors (physics-based)
density_cols = ['error_D_11','error_D_11H','error_D_14']
bm_cols = ['error_BM_T11','error_BM_T11H','error_BM_T14']
se_cols = ['error_SE_T11','error_SE_T11H','error_SE_T14']

df['density_err'] = df[density_cols].sum(axis=1)
df['bm_err'] = df[bm_cols].sum(axis=1)
df['se_err'] = df[se_cols].sum(axis=1)

In [ ]:
# Normalize + Smooth

# Normalize total error (per optimizer)
# df['norm_error'] = df.groupby('optimizers', sort=False)['total_error'].transform(lambda x: x / x.iloc[0])
df['norm_error'] = df.groupby('optimizers', sort=False)['total_error'].transform(lambda x: x / 1)

# Rolling smoothing (change window if needed)
window = 3

df['smooth_error'] = df.groupby('optimizers', sort=False)['norm_error'].transform(lambda x: x.rolling(window, min_periods=1).mean())

# Smooth grouped errors
for col in ['density_err', 'bm_err', 'se_err']:
    df[col + '_smooth'] = df.groupby('optimizers', sort=False)[col].transform(lambda x: x.rolling(window, min_periods=1).mean())

In [ ]:
plt.figure(figsize=(6,4))

for name, g in df.groupby('optimizers', sort=False):

    color = OPT_COLORS.get(name, 'black')

    if name == 'NSGA2':
        lw, alpha, z = 2.5, 1.0, 5   # strongest
    elif name == 'BO':
        lw, alpha, z = 2.0, 0.9, 4
    elif name == 'LHS':
        lw, alpha, z = 1.5, 0.7, 3
    else:  # Grid
        lw, alpha, z = 1.2, 0.5, 2
    plt.plot(
        g['iteration'],
        g['smooth_error'],
        color=color,
        linewidth=lw,
        alpha=alpha,
        label=name,
        zorder=z
    )
    # RAW (faint)
    # plt.plot(g['iteration'], g['total_error'],color=color, alpha=0.15, linewidth=1)
    
plt.xlabel("MD Simulation")
plt.ylabel("Total Error")

# clean grid (y-only → Nature style)
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("clean_convergence.png", dpi=300)
plt.show()

In [ ]:
def plot_convergence(df, column, ylabel, filename):

    plt.figure(figsize=(6,4))

    for name, g in df.groupby('optimizers', sort=False):

        color = OPT_COLORS.get(name, 'black')

        lw = 2.5 if name == 'NSGA2' else 2.0 if name == 'BO' else 1.5
        alpha = 1.0 if name in ['NSGA2','BO'] else 0.6

        plt.plot(g['iteration'], g[column],
                 color=color, linewidth=lw, alpha=alpha, label=name)

    plt.xlabel("MD Simulation")
    plt.ylabel(ylabel)
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.legend(frameon=False)

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_convergence(df, 'smooth_error', "Total Error", "total.png")
plot_convergence(df, 'density_err_smooth', "Density Error (%)", "density.png")
plot_convergence(df, 'bm_err_smooth', "Bulk Modulus Error (%)", "bm.png")
plot_convergence(df, 'se_err_smooth', "Surface Energy Error (%)", "se.png")

In [ ]:
df['iteration1'] = df.groupby('optimizers1', sort=False).cumcount() + 1


plt.figure(figsize=(10,4))

for name, g in df.groupby('optimizers1', sort=False):

    color = OPT_COLORS.get(name)

    # RAW (very faint)
    plt.plot(
        g['iteration1'],
        g['norm_error'],
        color=color,
        alpha=0.15,
        linewidth=1
    )

    # SMOOTH (main signal)
    lw = 2.5 if name == 'NSGA2' else 2.0 if name == 'BO' else 1.5
    z = 5 if name == 'NSGA2' else 4 if name == 'BO' else 3

    plt.plot(
        g['iteration1'],
        g['smooth_error'],
        color=color,
        linewidth=lw,
        label=name,
        zorder=z
    )

# Vertical separators (make subtle!)
for x in [60, 157, 267]:
    plt.axvline(x=x, color='gray', linewidth=1, alpha=0.6)

# put text horozontally: at x=30 put 'Grid', at x=108.5 put 'LHS', at x=212 put 'BO', at x=327 put 'NSGA2'
# ---------------------------
# SECTION LABELS (top aligned)
# ---------------------------
labels = [
    (30, 'Grid'),
    (108.5, 'LHS'),
    (212, 'BO'),
    (327, 'NSGA2')
]

y_pos = plt.ylim()[1] * 0.85  # slightly below top

for x, text in labels:
    plt.text(
        x, y_pos,
        text,
        ha='center',
        va='center',
        fontsize=11,
        fontweight='bold',
        color='black'
    )


regions = [
    (0, 60, 'Grid'),
    (60, 157, 'LHS'),
    (157, 267, 'BO'),
    (267, 387, 'NSGA2')
]

for start, end, name in regions:
    plt.axvspan(start, end, color=OPT_COLORS.get(name), alpha=0.05)
    
    
plt.xlim(0, 387)

plt.xlabel("MD Simulation Iteration")
plt.ylabel("Total Error")

# clean grid (y-only)
plt.grid(axis='y', linestyle='--', alpha=0.4)
# 
# plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("clean_convergence_overlay.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
plt.figure(figsize=(6,4))

bins = np.linspace(df['total_error'].min(), df['total_error'].max(), 30)

for name, g in df.groupby('optimizers', sort=False):

    color = OPT_COLORS.get(name, 'black')
    alpha = 0.8 if name == 'NSGA2' else 0.5 if name == 'BO' else 0.3

    plt.hist(
        g['total_error'],
        bins=bins,
        color=color,
        alpha=alpha,
        label=name,
        edgecolor='none'
    )

plt.xlabel("Total Error")
plt.ylabel("Frequency")

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("error_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

for name, g in df.groupby('optimizers', sort=False):

    color = OPT_COLORS.get(name, 'black')

    size = 40 if name == 'NSGA2' else 30
    alpha = 0.8 if name in ['NSGA2','BO'] else 0.4

    plt.scatter(
        g['density_err'],
        g['bm_err'],
        s=size,
        alpha=alpha,
        color=color,
        edgecolor='k',
        linewidth=0.2,
        label=name
    )

plt.xlabel("Density Error (%)")
plt.ylabel("Bulk Modulus Error (%)")

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("density_vs_bm.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

for name, g in df.groupby('optimizers', sort=False):

    color = OPT_COLORS.get(name, 'black')

    plt.scatter(
        g['density_err'],
        g['bm_err'],
        s=25,
        alpha=0.4,
        color=color
    )

# Best solutions
best = df.nsmallest(10, 'total_error')

plt.scatter(
    best['density_err'],
    best['bm_err'],
    s=120,
    marker='*',
    color='#6a3d9a',
    # edgecolor='k',
    # linewidth=0.5,
    label='Best solutions',
    zorder=10
)

plt.xlabel("Density Error (%)")
plt.ylabel("Bulk Modulus Error (%)")

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("pareto_highlight.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.boxplot(
    x='optimizers',
    y='total_error',
    data=df,
    palette=OPT_COLORS,
    width=0.6,
    linewidth=1.2,
    fliersize=3,
    medianprops=dict(color='black', linewidth=2)
)

sns.stripplot(
    x='optimizers',
    y='total_error',
    data=df,
    color='black',
    size=2,
    alpha=0.25
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ---------------------------
# MAIN FIGURE
# ---------------------------
fig, ax = plt.subplots(figsize=(6,4))

bins = np.linspace(df['total_error'].min(), df['total_error'].max(), 30)

for name, g in df.groupby('optimizers', sort=False):

    color = OPT_COLORS.get(name, 'black')
    alpha = 0.85 if name == 'NSGA2' else 0.55 if name == 'BO' else 0.35
    
    ax.hist(g['total_error'], bins=bins, color=color, alpha=alpha, label=name, edgecolor='none')

# ---------------------------
# MAIN AX STYLING
# ---------------------------
ax.set_xlabel("Total error")
ax.set_ylabel("Frequency")
# ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(frameon=False,ncol=2)

# ---------------------------
# INSET AXIS (BOXPLOT)
# ---------------------------
# [x, y, width, height] in figure fraction
inset = fig.add_axes([0.55, 0.32, 0.4, 0.4])

sns.boxplot(x='optimizers', y='total_error', data=df, palette=OPT_COLORS1, width=0.6, linewidth=1.0, fliersize=3, medianprops=dict(color='black', linewidth=1.0), ax=inset)
# sns.stripplot(x='optimizers', y='total_error', data=df, color='black', size=1.5, alpha=0.25,ax=inset)


# ---------------------------
# CLEAN INSET (VERY IMPORTANT)
# ---------------------------
inset.set_xlabel("")
inset.set_ylabel("Total error", fontsize=8)
inset.set_title("")

# remove clutter
inset.tick_params(axis='x', labelsize=7)
inset.tick_params(axis='y', labelsize=7)

# # optional: remove top/right spines
# for spine in ['top', 'right']:
#     inset.spines[spine].set_visible(False)
inset.patch.set_edgecolor('black')
inset.patch.set_linewidth(0.8)


# reduce inset axis line width
for spine in inset.spines.values():
    spine.set_linewidth(0.8)

inset.text(0.5, 1.05, "Distribution summary", transform=inset.transAxes, ha='center', fontsize=8)

# light grid
inset.grid(axis='y', linestyle='--', alpha=0.3)

# ---------------------------
# FINAL
# ---------------------------
plt.tight_layout()
plt.savefig("error_distribution_with_inset.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------
# PROPERTY LABELS
# ---------------------------
prop_dict = {}
for i, prop in enumerate([
    'Density (Tobermorite 11 Å Merlino)',
    'Density (Tobermorite 11 Å Hamid)',
    'Density (Tobermorite 14 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Hamid)',
    'Surface Energy (Tobermorite 14 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Hamid)',
    'Bulk Modulus (Tobermorite 14 Å Merlino)'
]):
    prop_dict[f'Property {i+1}'] = prop

# ---------------------------
# PARETO FUNCTION
# ---------------------------
def pareto_front(points):
    is_pareto = np.ones(points.shape[0], dtype=bool)
    for i, p in enumerate(points):
        if is_pareto[i]:
            is_pareto[is_pareto] = (
                np.any(points[is_pareto] < p, axis=1) |
                np.all(points[is_pareto] == p, axis=1)
            )
            is_pareto[i] = True
    return is_pareto

# ---------------------------
# DATA
# ---------------------------
error_cols = [
    'error_D_11', 'error_D_11H', 'error_D_14',
    'error_SE_T11', 'error_SE_T11H', 'error_SE_T14',
    'error_BM_T11', 'error_BM_T11H', 'error_BM_T14'
]

y_error_all = df[error_cols].values
optimizers = df['optimizers'].values

# thresholds
targeted_error = np.array([1, 1, 1, 10, 10, 10, 15, 15, 15])

# ---------------------------
# SELECTED PAIRS
# ---------------------------
selected_pairs = [
    (0, 3),
    (0, 6),
    (3, 6),
    (1, 4)
]

# ---------------------------
# PLOT
# ---------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (i, j) in zip(axes, selected_pairs):

    x = y_error_all[:, i]
    y = y_error_all[:, j]
    points = np.vstack([x, y]).T

    # ---------------------------
    # SCATTER (BY OPTIMIZER)
    # ---------------------------
    # for opt in np.unique(optimizers):
    for opt in ['Grid','LHS','BO','NSGA2']:
        mask_opt = (optimizers == opt)

        ax.scatter(
            x[mask_opt],
            y[mask_opt],
            s=20,
            alpha=0.6,
            color=OPT_COLORS.get(opt, 'gray'),
            label=opt if ax == axes[0] else None  # avoid duplicate legend
        )

    # ---------------------------
    # PARETO FRONT
    # ---------------------------
    mask = pareto_front(points)
    pareto_pts = points[mask]

    pareto_pts = pareto_pts[np.argsort(pareto_pts[:, 0])]
    _, unique_idx = np.unique(pareto_pts[:, 0], return_index=True)
    pareto_pts = pareto_pts[unique_idx]

    ax.plot(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        linewidth=2.5,
        label='Pareto front' if ax == axes[0] else None
    )

    ax.scatter(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        s=20
    )

    # ---------------------------
    # TARGET REGION
    # ---------------------------
    rect = plt.Rectangle(
        (0, 0),
        targeted_error[i],
        targeted_error[j],
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.08
    )
    ax.add_patch(rect)

    # # ---------------------------
    # # BEST SOLUTIONS (TOP-K GLOBAL)
    # # ---------------------------
    # k = 20
    # best_idx = np.argsort(df['total_error'].values)[:k]

    # ax.scatter(
    #     x[best_idx],
    #     y[best_idx],
    #     color='purple',
    #     s=90,
    #     marker='*',
    #     label='Best solutions' if ax == axes[0] else None,
    #     zorder=5
    # )

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_xlabel(prop_dict[f"Property {i+1}"])
    ax.set_ylabel(prop_dict[f"Property {j+1}"])

    ax.grid(True, linestyle='--', alpha=0.5)

    # # Optional zoom (recommended)
    # ax.set_xlim(0, x.max()+targeted_error[i]*1)
    # ax.set_ylim(0, y.max()+targeted_error[i]*1)

    # Optional zoom (recommended)
    # ax.set_xlim(0, min(x.max(), targeted_error[i]*3))
    # ax.set_ylim(0, min(y.max(), targeted_error[j]*3))

# ---------------------------
# LEGEND (ONLY ONCE)
# ---------------------------
axes[0].legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------
# PROPERTY LABELS
# ---------------------------
prop_dict = {}
for i, prop in enumerate([
    'Density (Tobermorite 11 Å Merlino)',
    'Density (Tobermorite 11 Å Hamid)',
    'Density (Tobermorite 14 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Hamid)',
    'Surface Energy (Tobermorite 14 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Hamid)',
    'Bulk Modulus (Tobermorite 14 Å Merlino)'
]):
    prop_dict[f'Property {i+1}'] = prop

# ---------------------------
# PARETO FUNCTION
# ---------------------------
def pareto_front(points):
    is_pareto = np.ones(points.shape[0], dtype=bool)
    for i, p in enumerate(points):
        if is_pareto[i]:
            is_pareto[is_pareto] = (
                np.any(points[is_pareto] < p, axis=1) |
                np.all(points[is_pareto] == p, axis=1)
            )
            is_pareto[i] = True
    return is_pareto

# ---------------------------
# DATA
# ---------------------------
error_cols = [
    'error_D_11', 'error_D_11H', 'error_D_14',
    'error_SE_T11', 'error_SE_T11H', 'error_SE_T14',
    'error_BM_T11', 'error_BM_T11H', 'error_BM_T14'
]

y_error_all = df[error_cols].values
optimizers = df['optimizers'].values

# thresholds
targeted_error = np.array([1, 1, 1, 10, 10, 10, 15, 15, 15])

# ---------------------------
# SELECTED PAIRS
# ---------------------------
selected_pairs = [
    (0, 3),
    (0, 6),
    (3, 6),
    (1, 4)
]

# ---------------------------
# PLOT
# ---------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (i, j) in zip(axes, selected_pairs):

    x = y_error_all[:, i]
    y = y_error_all[:, j]
    points = np.vstack([x, y]).T

    # ---------------------------
    # SCATTER (BY OPTIMIZER)
    # ---------------------------
    # for opt in np.unique(optimizers):
    for opt in ['Grid','LHS','BO','NSGA2']:
        mask_opt = (optimizers == opt)

        ax.scatter(
            x[mask_opt],
            y[mask_opt],
            s=20,
            alpha=0.6,
            color=OPT_COLORS.get(opt, 'gray'),
            label=opt if ax == axes[0] else None  # avoid duplicate legend
        )

    # ---------------------------
    # PARETO FRONT
    # ---------------------------
    mask = pareto_front(points)
    pareto_pts = points[mask]

    pareto_pts = pareto_pts[np.argsort(pareto_pts[:, 0])]
    _, unique_idx = np.unique(pareto_pts[:, 0], return_index=True)
    pareto_pts = pareto_pts[unique_idx]

    ax.plot(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        linewidth=2.5,
        label='Pareto front' if ax == axes[0] else None
    )

    ax.scatter(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        s=20
    )

    # ---------------------------
    # TARGET REGION
    # ---------------------------
    rect = plt.Rectangle(
        (0, 0),
        targeted_error[i],
        targeted_error[j],
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.08
    )
    ax.add_patch(rect)

    # # ---------------------------
    # # BEST SOLUTIONS (TOP-K GLOBAL)
    # # ---------------------------
    # k = 20
    # best_idx = np.argsort(df['total_error'].values)[:k]

    # ax.scatter(
    #     x[best_idx],
    #     y[best_idx],
    #     color='purple',
    #     s=90,
    #     marker='*',
    #     label='Best solutions' if ax == axes[0] else None,
    #     zorder=5
    # )

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_xlabel(prop_dict[f"Property {i+1}"])
    ax.set_ylabel(prop_dict[f"Property {j+1}"])

    ax.grid(True, linestyle='--', alpha=0.5)

    # # Optional zoom (recommended)
    # ax.set_xlim(0, x.max()+targeted_error[i]*1)
    # ax.set_ylim(0, y.max()+targeted_error[i]*1)

    # Optional zoom (recommended)
    ax.set_xlim(0, min(x.max(), targeted_error[i]*3))
    ax.set_ylim(0, min(y.max(), targeted_error[j]*3))

# ---------------------------
# LEGEND (ONLY ONCE)
# ---------------------------
axes[0].legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------
# PROPERTY LABELS
# ---------------------------
prop_dict = {}
for i, prop in enumerate([
    'Density (Tobermorite 11 Å Merlino)',
    'Density (Tobermorite 11 Å Hamid)',
    'Density (Tobermorite 14 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Hamid)',
    'Surface Energy (Tobermorite 14 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Hamid)',
    'Bulk Modulus (Tobermorite 14 Å Merlino)'
]):
    prop_dict[f'Property {i+1}'] = prop

# ---------------------------
# PARETO FUNCTION
# ---------------------------
def pareto_front(points):
    is_pareto = np.ones(points.shape[0], dtype=bool)
    for i, p in enumerate(points):
        if is_pareto[i]:
            is_pareto[is_pareto] = (
                np.any(points[is_pareto] < p, axis=1) |
                np.all(points[is_pareto] == p, axis=1)
            )
            is_pareto[i] = True
    return is_pareto

# ---------------------------
# DATA
# ---------------------------
error_cols = [
    'error_D_11', 'error_D_11H', 'error_D_14',
    'error_SE_T11', 'error_SE_T11H', 'error_SE_T14',
    'error_BM_T11', 'error_BM_T11H', 'error_BM_T14'
]

y_error_all = df[error_cols].values
optimizers = df['optimizers'].values

# thresholds
targeted_error = np.array([1, 1, 1, 10, 10, 10, 15, 15, 15])

# ---------------------------
# SELECTED PAIRS
# ---------------------------
selected_pairs = [(0, 3), (0, 6)]

# ---------------------------
# PLOT
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes = axes.flatten()

for ax, (i, j) in zip(axes, selected_pairs):

    x = y_error_all[:, i]
    y = y_error_all[:, j]
    points = np.vstack([x, y]).T

    # ---------------------------
    # SCATTER (BY OPTIMIZER)
    # ---------------------------
    # for opt in np.unique(optimizers):
    for opt in ['Grid','LHS','BO','NSGA2']:
        mask_opt = (optimizers == opt)

        ax.scatter(
            x[mask_opt],
            y[mask_opt],
            s=20,
            alpha=0.6,
            color=OPT_COLORS.get(opt, 'gray'),
            label=opt if ax == axes[0] else None  # avoid duplicate legend
        )

    # ---------------------------
    # PARETO FRONT
    # ---------------------------
    mask = pareto_front(points)
    pareto_pts = points[mask]

    pareto_pts = pareto_pts[np.argsort(pareto_pts[:, 0])]
    _, unique_idx = np.unique(pareto_pts[:, 0], return_index=True)
    pareto_pts = pareto_pts[unique_idx]

    ax.plot(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        linewidth=2.5,
        label='Pareto front' if ax == axes[0] else None
    )

    ax.scatter(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        s=20
    )

    # ---------------------------
    # TARGET REGION
    # ---------------------------
    rect = plt.Rectangle(
        (0, 0),
        targeted_error[i],
        targeted_error[j],
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.08
    )
    ax.add_patch(rect)

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_xlabel(prop_dict[f"Property {i+1}"])
    ax.set_ylabel(prop_dict[f"Property {j+1}"])
    ax.grid(True, linestyle='--', alpha=0.5)

    # Optional zoom (recommended)
    ax.set_xlim(0, min(x.max(), targeted_error[i]*3))
    ax.set_ylim(0, min(y.max(), targeted_error[j]*3))

# ---------------------------
# LEGEND (ONLY ONCE)
# ---------------------------
axes[0].legend()
# axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:

from matplotlib.patches import Rectangle
from sklearn.linear_model import LinearRegression
from itertools import combinations
from sklearn.metrics import r2_score

# Setup
property_indices = list(range(9))
prop_pairs = list(combinations(property_indices, 2))  # 9C2 = 36
fig, axs = plt.subplots(6, 6, figsize=(15, 15))
axs = axs.flatten()

for idx, (i, j) in enumerate(prop_pairs):
    ax = axs[idx]
    
    # Data for the current property pair
    x = y_error_all[:, i].reshape(-1, 1)
    y = y_error_all[:, j]

    # Scatter plot
    ax.scatter(x, y, s=10, alpha=0.7, label='Data')
    
    # Linear regression
    model = LinearRegression()
    model.fit(x, y)
    y_pred = model.predict(x)
    r2 = r2_score(y, y_pred)
    slope = model.coef_[0]
    intercept = model.intercept_
    
    # Plot the linear fit
    x_vals = np.linspace(x.min(), x.max(), 100).reshape(-1, 1)
    y_vals = model.predict(x_vals)
    ax.plot(x_vals, y_vals, color='red', linewidth=1, label='Linear Fit')

    # Axis labels
    ax.set_xlabel(prop_dict[f"Property {i+1}"], fontsize=8)
    ax.set_ylabel(prop_dict[f"Property {j+1}"], fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
   
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    # Add red box for target error zone
    rect = Rectangle(
        (0, 0),  # bottom left corner
        targeted_error[i],  # width
        targeted_error[j],  # height
        linewidth=1,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

    # Annotation with equation and R²
    ax.text(0.05, 0.95,
            f'y = {slope:.2f}x + {intercept:.2f}\nR² = {r2:.2f}',
            transform=ax.transAxes,
            fontsize=8,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()    
plt.show()

In [ ]:
df[['sc_r','sc_eps','oc_r','oc_eps','error_D_11', 'error_D_11H', 'error_D_14', 'error_SE_T11', 'error_SE_T11H', 'error_SE_T14', 'error_BM_T11', 'error_BM_T11H', 'error_BM_T14']]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# ---------------------------
# INPUT
# ---------------------------
# df = your dataframe

x = df['sc_r'].values
y = df['oc_r'].values

# All 9 error columns
error_cols = ['error_D_11', 'error_D_11H', 'error_D_14','error_SE_T11', 'error_SE_T11H', 'error_SE_T14','error_BM_T11', 'error_BM_T11H', 'error_BM_T14']


col2name = {
    'error_D_11': 'Density (Tobermorite 11 Å Merlino)',
    'error_D_11H': 'Density (Tobermorite 11 Å Hamid)',
    'error_D_14': 'Density (Tobermorite 14 Å Merlino)',
    'error_SE_T11': 'Surface Energy (Tobermorite 11 Å Merlino)',
    'error_SE_T11H': 'Surface Energy (Tobermorite 11 Å Hamid)',
    'error_SE_T14': 'Surface Energy (Tobermorite 14 Å Merlino)',
    'error_BM_T11': 'Bulk Modulus (Tobermorite 11 Å Merlino)',
    'error_BM_T11H': 'Bulk Modulus (Tobermorite 11 Å Hamid)',
    'error_BM_T14': 'Bulk Modulus (Tobermorite 14 Å Merlino)'
}


# ---------------------------
# GRID (important for smooth plots)
# ---------------------------
grid_res = 200  # increase for smoother plots

xi = np.linspace(x.min(), x.max(), grid_res)
yi = np.linspace(y.min(), y.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

# ---------------------------
# PLOTTING
# ---------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for i, col in enumerate(error_cols):
    ax = axes[i // 3, i % 3]

    z = df[col].values

    # Interpolate scattered data → grid
    Zi = griddata((x, y), z, (Xi, Yi), method='linear')

    # Optional: mask bad regions (outside convex hull)
    Zi = np.ma.masked_invalid(Zi)

    # Plot
    contour = ax.contourf(Xi, Yi, Zi,levels=20,cmap='viridis')

    # Scatter original points (optional but useful)
    ax.scatter(x, y, c='white', s=10, alpha=0.6)

    # Titles
    ax.set_title(f'{col2name[col]}', fontsize=10)

    # Labels
    ax.set_xlabel('Si (σ)')
    ax.set_ylabel('O (σ)')

    # Colorbar
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label('Error percentage')

# ---------------------------
# Layout
# ---------------------------
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# ---------------------------
# INPUT
# ---------------------------
x = df['sc_r'].values
y = df['oc_r'].values

error_cols = ['error_D_11', 'error_D_11H', 'error_D_14', 'error_SE_T11', 'error_SE_T11H', 'error_SE_T14', 'error_BM_T11', 'error_BM_T11H', 'error_BM_T14']

# ---------------------------
# GRID
# ---------------------------
grid_res = 200
xi = np.linspace(x.min(), x.max(), grid_res)
yi = np.linspace(y.min(), y.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

# ---------------------------
# PLOTTING
# ---------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for i, col in enumerate(error_cols):
    ax = axes[i // 3, i % 3]

    z = df[col].values

    # Interpolation
    Zi = griddata((x, y), z, (Xi, Yi), method='linear')
    Zi = np.ma.masked_invalid(Zi)

    # ---------------------------
    # MAIN CONTOUR
    # ---------------------------
    contour = ax.contourf(Xi, Yi, Zi, levels=20, cmap='viridis')

    # Scatter
    ax.scatter(x, y, c='white', s=10, alpha=0.6)

    # ---------------------------
    # THRESHOLD PER PROPERTY
    # ---------------------------
    if "error_D" in col:
        threshold = 1.0
        _alpha = 0.1
    elif "error_SE" in col:
        threshold = 10.0
        _alpha = 0.1
    elif "error_BM" in col:
        threshold = 15.0
        _alpha = 0.15
        

    # ---------------------------
    # MASK (per subplot)
    # ---------------------------
    mask = Zi < threshold

    # ---------------------------
    # HIGHLIGHT REGION
    # ---------------------------
    ax.contourf(Xi, Yi, mask, levels=[0.5, 1], colors=['red'], alpha=_alpha)

    ax.contour(Xi, Yi, mask, levels=[0.5], colors='red', linewidths=0.5)

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_title(f'Loss Landscape for {col}', fontsize=10)
    ax.set_xlabel('Si (σ)')
    ax.set_ylabel('O (σ)')

    # Colorbar
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label('% error')

# ---------------------------
# FINAL LAYOUT
# ---------------------------
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# ---------------------------
# INPUT
# ---------------------------
x = df['sc_r'].values
y = df['oc_r'].values

error_cols = ['error_D_11', 'error_D_11H', 'error_D_14', 'error_SE_T11', 'error_SE_T11H', 'error_SE_T14', 'error_BM_T11', 'error_BM_T11H', 'error_BM_T14']

# ---------------------------
# GRID
# ---------------------------
grid_res = 200
xi = np.linspace(x.min(), x.max(), grid_res)
yi = np.linspace(y.min(), y.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

# ---------------------------
# PLOTTING
# ---------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for i, col in enumerate(error_cols):
    ax = axes[i // 3, i % 3]

    z = df[col].values

    # Interpolation
    Zi = griddata((x, y), z, (Xi, Yi), method='linear')
    Zi = np.ma.masked_invalid(Zi)

    # ---------------------------
    # MAIN CONTOUR
    # ---------------------------
    # contour = ax.contourf(Xi, Yi, Zi, levels=20, cmap='viridis')
    
    # define row-wise limits
    row = i // 3

    if row == 0:
        vmin, vmax = 0, 25   # Density
    elif row == 1:
        vmin, vmax = 0, 90   # SE
    else:
        vmin, vmax = 0, 55   # BM

    contour = ax.contourf(Xi, Yi, Zi, levels=20, cmap='viridis', vmin=vmin, vmax=vmax)


    # Scatter
    ax.scatter(x, y, c='white', s=10, alpha=0.6)

    # ---------------------------
    # THRESHOLD PER PROPERTY
    # ---------------------------
    if "error_D" in col:
        threshold = 1.0
        _alpha = 0.05
    elif "error_SE" in col:
        threshold = 10.0
        _alpha = 0.05
    elif "error_BM" in col:
        threshold = 15.0
        _alpha = 0.05
        

    # ---------------------------
    # MASK (per subplot)
    # ---------------------------
    mask = Zi < threshold

    # ---------------------------
    # HIGHLIGHT REGION
    # ---------------------------
    ax.contourf(Xi, Yi, mask, levels=[0.5, 1], colors=['red'], alpha=_alpha)

    ax.contour(Xi, Yi, mask, levels=[0.5], colors='red', linewidths=0.5)

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_title(f'Loss Landscape for {col}', fontsize=10)
    ax.set_xlabel('Si (σ)')
    ax.set_ylabel('O (σ)')

    # Colorbar
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label('% error')

# ---------------------------
# FINAL LAYOUT
# ---------------------------
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# ---------------------------
# INPUT
# ---------------------------
x = df['sc_r'].values
y = df['oc_r'].values

error_cols = ['error_D_11', 'error_D_11H', 'error_D_14', 'error_SE_T11', 'error_SE_T11H', 'error_SE_T14', 'error_BM_T11', 'error_BM_T11H', 'error_BM_T14']

# ---------------------------
# GRID
# ---------------------------
grid_res = 200
xi = np.linspace(x.min(), x.max(), grid_res)
yi = np.linspace(y.min(), y.max(), grid_res)
Xi, Yi = np.meshgrid(xi, yi)

# ---------------------------
# ROW-WISE CONSTRAINTS
# ---------------------------
thresholds = [1.0, 10.0, 15.0]
row_masks = []

for row_idx in range(3):
    cols = error_cols[row_idx*3:(row_idx+1)*3]

    # STRICT condition → all 3 must satisfy
    row_metric = df[cols].max(axis=1)

    Zi_mask = griddata((x, y), row_metric, (Xi, Yi), method='linear')
    Zi_mask = np.ma.masked_invalid(Zi_mask)

    mask = Zi_mask < thresholds[row_idx]
    row_masks.append(mask)

# ---------------------------
# PLOTTING
# ---------------------------
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for i, col in enumerate(error_cols):
    ax = axes[i // 3, i % 3]

    z = df[col].values

    # Interpolation (NO smoothing — as you want)
    Zi = griddata((x, y), z, (Xi, Yi), method='linear')
    Zi = np.ma.masked_invalid(Zi)

    # Main contour
    contour = ax.contourf(Xi, Yi, Zi, levels=20, cmap='viridis')

    # Scatter points
    ax.scatter(x, y, c='white', s=10, alpha=0.6)

    # ---------------------------
    # ADD FEASIBLE REGION
    # ---------------------------
    mask = row_masks[i // 3]

    if "error_BM" in col:
        _alpha = 0.15
    else:
        _alpha = 0.1

    # Filled region
    ax.contourf(Xi, Yi, mask, levels=[0.5, 1], colors=['red'], alpha=_alpha)
    # Boundary line
    ax.contour(Xi, Yi, mask, levels=[0.5], colors='red', linewidths=0.5)

    # Labels
    ax.set_title(f'Loss Landscape for {col}', fontsize=10)
    ax.set_xlabel('Si (σ)')
    ax.set_ylabel('O (σ)')

    # Colorbar
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label('% error')

# ---------------------------
# FINAL LAYOUT
# ---------------------------
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

prop_names = [
    'Density (11Å)',
    'Density (11Å H)',
    'Density (14Å)',
    'SE (11Å)',
    'SE (11Å H)',
    'SE (14Å)',
    'BM (11Å)',
    'BM (11Å H)',
    'BM (14Å)'
]

# ---------------------------
# 1. DEFINE MODEL (IMPORTANT)
# ---------------------------
class FlexibleNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, dropout_p=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ---------------------------
# 2. LOAD DATA
# ---------------------------
df = pd.read_csv('../../../data/training/training_data_145.csv')
df = df.dropna()

X = df.values[:, :4]
y = df.values[:, 4:]

# same split as training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 3. SCALING (MUST MATCH TRAINING)
# ---------------------------
X_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)

X_test_scaled = X_scaler.transform(X_test)

# ---------------------------
# 4. LOAD MODEL
# ---------------------------
model_path = '../../../data/training/models/NSGA2/model_9.pth'

model = FlexibleNN(input_dim=4, hidden_layers=[64, 32], output_dim=9, dropout_p=0.3)
model.load_state_dict(torch.load(model_path))
model.eval()

# ---------------------------
# 5. PREDICTION
# ---------------------------
with torch.no_grad():
    y_pred_scaled = model(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()

y_pred = y_scaler.inverse_transform(y_pred_scaled)

# ---------------------------
# 6. PROPERTY NAMES
# ---------------------------
prop_names = [
    'D11', 'D11H', 'D14',
    'SE_T11', 'SE_T11H', 'SE_T14',
    'BM_T11', 'BM_T11H', 'BM_T14'
]

# ---------------------------
# 7. PLOT (3x3 GRID)
# ---------------------------
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):

    y_true = y_test[:, i]
    y_p = y_pred[:, i]

    # Scatter
    ax.scatter(y_true, y_p, s=20, alpha=0.7)

    # Ideal line
    min_val = min(y_true.min(), y_p.min())
    max_val = max(y_true.max(), y_p.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')

    # R2
    r2 = r2_score(y_true, y_p)

    ax.set_title(f"{prop_names[i]} (R²={r2:.2f})", fontsize=10)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")

    ax.grid(True, linestyle='--', alpha=0.4)

ax.set_aspect('equal', adjustable='box')
ax.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'r:', alpha=0.3)
ax.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'r:', alpha=0.3)


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# ---------------------------
# MIN-MAX NORMALIZATION (0 → 1)
# ---------------------------
scaler = MinMaxScaler()

y_test_norm = scaler.fit_transform(y_test)
y_pred_norm = scaler.transform(y_pred)  # use SAME scaler!

# ---------------------------
# COLOR GROUPS (3 × 3 shades)
# ---------------------------
colors = [
    # Density (blue shades)
    '#9ecae1', '#4292c6', '#084594',

    # Surface Energy (green shades)
    '#a1d99b', '#41ab5d', '#005a32',

    # Bulk Modulus (orange/red shades)
    '#fdae6b', '#e6550d', '#7f2704'
]

# ---------------------------
# LABELS
# ---------------------------
prop_names = [
    'D 11 Å', 'D 11 Å H', 'D 14 Å',
    'SE 11 Å', 'SE 11 Å H', 'SE 14 Å',
    'BM 11 Å', 'BM 11 Å H', 'BM 14 Å']

# ---------------------------
# PLOT
# ---------------------------
plt.figure(figsize=(5,5))

for i in range(9):
    r2 = r2_score(y_test[:, i], y_pred[:, i])  # use ORIGINAL scale

    plt.scatter(
        y_test_norm[:, i],
        y_pred_norm[:, i],
        color=colors[i],
        # alpha=0.7,
        s=40,
        edgecolor='k',
        linewidth=0.2,
        label=f"{prop_names[i]} [R²={r2:.2f}]"
    )

# Ideal line (0–1)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.0, alpha=0.7)

# ---------------------------
# STYLING
# ---------------------------
plt.xlabel("True (normalized)")
plt.ylabel("Predicted (normalized)")
# plt.title("Neural Network Property Prediction Performance", fontsize=12)
# plt.title("Overall Model Performance (MinMax Normalized)")
# plt.grid(alpha=0.3)

# plt.xlim(0, 1)
# plt.ylim(0, 1)

# reduce the spacing between legend symbol and text
plt.legend(ncol=3, fontsize=7, frameon=True, columnspacing=0.1, handletextpad=0.1)


plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams, font_manager as fm
from scipy.interpolate import griddata
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# =========================================================
# 1. GLOBAL STYLE (CRITICAL — consistency)
# =========================================================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True
})

OPT_COLORS = {
    'NSGA2': '#d62728',
    'BO': '#1f77b4',
    'LHS': '#2ca02c',
    'Grid': '#7f7f7f'
}

OPT_COLORS_SOFT = {
    'NSGA2': '#DC4748',
    'BO': '#84B4D5',
    'LHS': '#B5DDB5',
    'Grid': '#D2D2D2'
}

# =========================================================
# 2. LOAD + PREP DATA
# =========================================================

df = pd.read_csv("../../../data/reference_results/error_data_pcff_all_wi_SE.csv")
df = df.loc[:, ~df.columns.str.contains('Unnamed')]

df['iteration'] = df.groupby('optimizers', sort=False).cumcount() + 1

# total error
df['total_error'] = df.filter(like='error_').sum(axis=1)

# grouped errors
df['density_err'] = df[['error_D_11','error_D_11H','error_D_14']].sum(axis=1)
df['bm_err'] = df[['error_BM_T11','error_BM_T11H','error_BM_T14']].sum(axis=1)
df['se_err'] = df[['error_SE_T11','error_SE_T11H','error_SE_T14']].sum(axis=1)

# smoothing
window = 3
df['smooth_error'] = df.groupby('optimizers')['total_error'] \
    .transform(lambda x: x.rolling(window, min_periods=1).mean())

# =========================================================
# FIG (d) — Convergence
# =========================================================
plt.figure(figsize=(5.5,4))

for name, g in df.groupby('optimizers', sort=False):

    lw = 2.5 if name=='NSGA2' else 2.0 if name=='BO' else 1.5
    alpha = 1.0 if name in ['NSGA2','BO'] else 0.5

    plt.plot(
        g['iteration'], g['smooth_error'],
        color=OPT_COLORS[name],
        linewidth=lw,
        alpha=alpha,
        label=name
    )

plt.xlabel("MD simulation")
plt.ylabel("Total error")
plt.grid(axis='y', alpha=0.4)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("fig_d_convergence.png", dpi=300)
plt.show()

# =========================================================
# FIG (e) — Distribution + inset
# =========================================================
fig, ax = plt.subplots(figsize=(5,4))

bins = np.linspace(df['total_error'].min(), df['total_error'].max(), 30)

for name, g in df.groupby('optimizers', sort=False):

    alpha = 0.85 if name=='NSGA2' else 0.55 if name=='BO' else 0.35

    ax.hist(g['total_error'], bins=bins, color=OPT_COLORS[name], alpha=alpha, label=name) #, edgecolor='none')

ax.set_xlabel("Total error")
ax.set_ylabel("Frequency")
ax.legend(frameon=False, ncol=2)
# ax.grid(axis='y', linestyle='--', alpha=0.4)

# inset
inset = fig.add_axes([0.52, 0.32, 0.4, 0.4])

sns.boxplot(x='optimizers', y='total_error', data=df, palette=OPT_COLORS_SOFT, linewidth=1, fliersize=3, ax=inset)
# sns.boxplot(x='optimizers', y='total_error', data=df, palette=OPT_COLORS1, width=0.6, linewidth=1.0, fliersize=3, medianprops=dict(color='black', linewidth=1.0), ax=inset)
# sns.stripplot(x='optimizers', y='total_error', data=df, color='black', size=1.5, alpha=0.25,ax=inset)

inset.set_xlabel("")
inset.set_ylabel("Total error", fontsize=8)
inset.tick_params(labelsize=7)

inset.patch.set_edgecolor('black')
inset.patch.set_linewidth(0.8)

# reduce inset axis line width
for spine in inset.spines.values():
    spine.set_linewidth(0.8)
    
inset.text(0.5, 1.05, "Distribution summary", transform=inset.transAxes, ha='center', fontsize=8)
inset.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("fig_e_distribution.png", dpi=300)
plt.show()


# =========================================================
# FIG (f) — Loss landscape
# =========================================================
x, y = df['sc_r'].values, df['oc_r'].values
z = df['error_SE_T11H'].values

xi = np.linspace(x.min(), x.max(), 200)
yi = np.linspace(y.min(), y.max(), 200)
Xi, Yi = np.meshgrid(xi, yi)

Zi = griddata((x, y), z, (Xi, Yi), method='linear')
Zi = np.ma.masked_invalid(Zi)

plt.figure(figsize=(5,4))
contour = plt.contourf(Xi, Yi, Zi, levels=20, cmap='viridis') # MAIN CONTOUR
plt.scatter(x, y, c='white', s=10, alpha=0.6) # Scatter

# highlight feasible region
mask = Zi < 10.0
plt.contourf(Xi, Yi, mask, levels=[0.5, 1], colors=['red'], alpha=0.1)
plt.contour(Xi, Yi, mask, levels=[0.5], colors='red', linewidths=0.5)

plt.xlabel('Si (σ)')
plt.ylabel('O (σ)')

cbar = plt.colorbar(contour)
cbar.set_label('Error percentage')

plt.tight_layout()
plt.savefig("fig_f_landscape.png", dpi=300)
plt.show()


# =========================================================
# FIG (g) — NN performance
# =========================================================
scaler = MinMaxScaler()
y_test_norm = scaler.fit_transform(y_test)
y_pred_norm = scaler.transform(y_pred)

colors = [
    '#9ecae1','#4292c6','#084594',
    '#a1d99b','#41ab5d','#005a32',
    '#fdae6b','#e6550d','#7f2704'
]

prop_names = [
    'D 11 Å', 'D 11 Å H', 'D 14 Å',
    'SE 11 Å', 'SE 11 Å H', 'SE 14 Å',
    'BM 11 Å', 'BM 11 Å H', 'BM 14 Å']


plt.figure(figsize=(5,4))

for i in range(9):
    r2 = r2_score(y_test[:,i], y_pred[:,i])
    plt.scatter(y_test_norm[:,i], y_pred_norm[:,i],
                color=colors[i], s=30,
                label=f"{prop_names[i]} [R²={r2:.2f}]")

plt.plot([0,1],[0,1],'k--', lw=1, alpha=0.7)

plt.xlabel("True (normalized)")
plt.ylabel("Predicted (normalized)")
plt.legend(ncol=3, fontsize=7, frameon=True, columnspacing=0.1, handletextpad=0.1)

plt.tight_layout()
plt.savefig("fig_g_nn.png", dpi=300)
plt.show()

# =========================================================
# FIG (h) — Pareto
# =========================================================


# ---------------------------
# PROPERTY LABELS
# ---------------------------
prop_dict = {}
for i, prop in enumerate([
    'Density (Tobermorite 11 Å Merlino)',
    'Density (Tobermorite 11 Å Hamid)',
    'Density (Tobermorite 14 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Merlino)',
    'Surface Energy (Tobermorite 11 Å Hamid)',
    'Surface Energy (Tobermorite 14 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Merlino)',
    'Bulk Modulus (Tobermorite 11 Å Hamid)',
    'Bulk Modulus (Tobermorite 14 Å Merlino)'
]):
    prop_dict[f'Property {i+1}'] = prop

# ---------------------------
# PARETO FUNCTION
# ---------------------------
def pareto_front(points):
    is_pareto = np.ones(points.shape[0], dtype=bool)
    for i, p in enumerate(points):
        if is_pareto[i]:
            is_pareto[is_pareto] = (
                np.any(points[is_pareto] < p, axis=1) |
                np.all(points[is_pareto] == p, axis=1)
            )
            is_pareto[i] = True
    return is_pareto


# ---------------------------
# DATA
# ---------------------------
error_cols = [
    'error_D_11', 'error_D_11H', 'error_D_14',
    'error_SE_T11', 'error_SE_T11H', 'error_SE_T14',
    'error_BM_T11', 'error_BM_T11H', 'error_BM_T14'
]

y_error_all = df[error_cols].values
optimizers = df['optimizers'].values

# thresholds
targeted_error = np.array([1, 1, 1, 10, 10, 10, 15, 15, 15])

# ---------------------------
# SELECTED PAIRS
# ---------------------------
selected_pairs = [(0, 3), (0, 6)]

fig, axes = plt.subplots(1,2, figsize=(9,4))

for ax, (i,j) in zip(axes, selected_pairs):
    x = y_error_all[:, i]
    y = y_error_all[:, j]
    points = np.vstack([x, y]).T

    # ---------------------------
    # SCATTER (BY OPTIMIZER)
    # ---------------------------
    # for opt in np.unique(optimizers):
    for opt in ['Grid','LHS','BO','NSGA2']:
        mask_opt = (optimizers == opt)

        ax.scatter(
            x[mask_opt],
            y[mask_opt],
            s=20,
            alpha=0.6,
            color=OPT_COLORS.get(opt, 'gray'),
            label=opt if ax == axes[0] else None  # avoid duplicate legend
        )

    # ---------------------------
    # PARETO FRONT
    # ---------------------------
    mask = pareto_front(points)
    pareto_pts = points[mask]

    pareto_pts = pareto_pts[np.argsort(pareto_pts[:, 0])]
    _, unique_idx = np.unique(pareto_pts[:, 0], return_index=True)
    pareto_pts = pareto_pts[unique_idx]

    ax.plot(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        linewidth=2.5,
        label='Pareto front' if ax == axes[0] else None
    )

    ax.scatter(
        pareto_pts[:, 0],
        pareto_pts[:, 1],
        color='black',
        s=20
    )

    # ---------------------------
    # TARGET REGION
    # ---------------------------
    rect = plt.Rectangle(
        (0, 0),
        targeted_error[i],
        targeted_error[j],
        linewidth=2,
        edgecolor='red',
        facecolor='red',
        alpha=0.08
    )
    ax.add_patch(rect)

    # ---------------------------
    # LABELS
    # ---------------------------
    ax.set_xlabel(prop_dict[f"Property {i+1}"])
    ax.set_ylabel(prop_dict[f"Property {j+1}"])
    ax.grid(True, alpha=0.4)

    # Optional zoom (recommended)
    ax.set_xlim(0, min(x.max(), targeted_error[i]*3))
    ax.set_ylim(0, min(y.max(), targeted_error[j]*3))

# ---------------------------
# LEGEND (ONLY ONCE)
# ---------------------------
axes[0].legend()
# axes[1].legend()

plt.tight_layout()
plt.savefig("fig_h_pareto.png", dpi=300)
plt.show()

